# CF-PE-CIR theo runbook E2
Notebook này giữ CLIP ViT-L/14 đóng băng và tách rõ B0, B1, B4, B5. Chế độ `smoke` chỉ kiểm tra kỹ thuật; chế độ `screening` mới tạo 20K–50K pseudo-edits. Không dùng triplet CIRR/Fashion-IQ để train.


In [ ]:
MODE = "screening"  # real CC3M data: target 20K-50K pseudo-edits
AUDIT_APPROVED = False  # chỉ đặt True sau khi đã xem mining_audit.csv
RUN_B0_SCHEDULE = False  # bật nếu cần chạy lại B0 1K -> 10K -> 100K
RUN_B5_AFTER_GO = True
USE_GOOGLE_DRIVE = True  # persist data and checkpoints
RUN_NAME = "screening_01"  # use a new name for each training experiment
SETTINGS = {
    "smoke": {"num_shards": 2, "images_per_shard": 2000, "min_tuples": 20, "max_tuples": 1000, "steps": 20},
    "screening": {"num_shards": 20, "images_per_shard": 5000, "min_tuples": 20000, "max_tuples": 50000, "steps": 2000},
}[MODE]
print(MODE, SETTINGS)


## 1. Nạp code và kiểm thử
Notebook tải trực tiếp phiên bản mới nhất từ nhánh `main`, sau đó cài dependency và chạy test.


In [ ]:
from pathlib import Path
import csv, json, os, shutil, subprocess, sys
repo = Path("/content/PE-CIR")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--depth", "1",
    "https://github.com/huylenhat1701-stack/PE-CIR.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]", "huggingface_hub"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## 2. GPU và Google Drive
Mặc định lưu dữ liệu và checkpoint tại `MyDrive/CF-PE-CIR`. Cho phép kết nối Google Drive khi được hỏi.


In [ ]:
import torch
assert torch.cuda.is_available(), "Hãy chọn Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive/CF-PE-CIR")
else:
    drive_root = Path("/content/CF-PE-CIR-output")
data_root = drive_root / "data"
runs_root = drive_root / "runs"
data_root.mkdir(parents=True, exist_ok=True)
runs_root.mkdir(parents=True, exist_ok=True)
print("Thư mục kết quả:", drive_root)


## 3. Tải ảnh và caption CC3M thật
Screening tải 20 shard, tối đa 5.000 ảnh/shard (100.000 ảnh đầu vào). Đây là tập con CC3M, không phải toàn bộ dữ liệu. Smoke cũng dùng ảnh thật nhưng chỉ lấy mẫu nhỏ.

Cần dung lượng trống trên Drive cho ảnh và trên Colab cho cache tải shard. Nếu mining chưa đủ 20.000 mẫu, tăng `num_shards` trong SETTINGS rồi chạy lại. Nhãn chỉnh sửa được tạo tự động từ caption, cần kiểm tra thủ công.


In [ ]:
from huggingface_hub import hf_hub_download, HfApi
revision = HfApi().dataset_info("pixparse/cc3m-wds").sha
source_root = data_root / ("cc3m_%s_%s" % (MODE, SETTINGS["images_per_shard"]))
source_root.mkdir(parents=True, exist_ok=True)
image_root = source_root / "images"
manifest_dir = source_root / "manifests"
image_root.mkdir(exist_ok=True)
manifest_dir.mkdir(exist_ok=True)
for shard_id in range(SETTINGS["num_shards"]):
    filename = f"cc3m-train-{shard_id:04d}.tar"
    manifest = manifest_dir / f"{shard_id:04d}.csv"
    if manifest.is_file():
        with manifest.open(encoding="utf-8") as stream:
            cached = list(csv.DictReader(stream))
        if cached and all((image_root / row["image"]).is_file() for row in cached):
            print("Reuse:", filename, "images:", len(cached), flush=True)
            continue
    print("Download / extract:", filename, flush=True)
    archive = Path(hf_hub_download(repo_id="pixparse/cc3m-wds", repo_type="dataset",
        filename=filename, revision=revision, cache_dir="/content/hf-cache"))
    subprocess.run([sys.executable, "scripts/prepare_cc3m_shard.py", str(archive),
        "--output-dir", str(image_root), "--manifest", str(manifest),
        "--max-images", str(SETTINGS["images_per_shard"])], check=True)
combined = source_root / "cc3m_captioned.csv"
rows = []
for shard_id in range(SETTINGS["num_shards"]):
    path = manifest_dir / f"{shard_id:04d}.csv"
    with path.open(encoding="utf-8") as stream:
        rows.extend(csv.DictReader(stream))
with combined.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["image", "caption"])
    writer.writeheader(); writer.writerows(rows)
captioned = sum(bool(row["caption"].strip()) for row in rows)
print("Ảnh:", len(rows), "có caption:", captioned, "revision:", revision)


## 4. Mine pseudo-edits, CF-A và CF-B
Mục tiêu: 20.000–50.000 mẫu. Nếu thiếu, tăng `num_shards` rồi chạy lại tải và mining. Shard đã hoàn tất được dùng lại.

Xem ít nhất 500 dòng audit và đối chiếu ảnh bằng ô bên dưới. Positive giữ Preserve và thỏa Edit; CF-A giữ Preserve nhưng sai Edit; CF-B thỏa Edit nhưng sai Preserve. CSV được lưu tại `MyDrive/CF-PE-CIR/data/pseudo_screening/mining_audit.csv`.


In [ ]:
pseudo_dir = data_root / f"pseudo_{MODE}"
pseudo_dir.mkdir(exist_ok=True)
all_tuples = pseudo_dir / "all.csv"
command = [sys.executable, "scripts/mine_pseudo_edits.py", "--source-manifest", str(combined),
    "--output", str(all_tuples), "--audit", str(pseudo_dir / "mining_audit.csv"),
    "--config-output", str(pseudo_dir / "mining_config.json"),
    "--min-tuples", str(SETTINGS["min_tuples"]), "--max-tuples", str(SETTINGS["max_tuples"]), "--seed", "0"]
if MODE == "smoke": command.append("--allow-small")
subprocess.run(command, check=True)
with all_tuples.open(encoding="utf-8") as stream:
    tuples = list(csv.DictReader(stream))
split = max(1, int(len(tuples) * 0.9))
for name, subset in (("train.csv", tuples[:split]), ("val.csv", tuples[split:])):
    with (pseudo_dir / name).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(tuples[0])); writer.writeheader(); writer.writerows(subset)
print("train:", split, "val:", len(tuples) - split, "audit:", pseudo_dir / "mining_audit.csv")
AUDIT_APPROVED = False  # review each newly mined dataset


In [ ]:
from IPython.display import display, HTML
from html import escape
from PIL import Image

audit_path = pseudo_dir / "mining_audit.csv"
with audit_path.open(encoding="utf-8") as stream:
    audit_rows = list(csv.DictReader(stream))
print("Audit CSV:", audit_path, "Rows:", len(audit_rows))
AUDIT_START = 0  # change to 50, 100, ... for the next page
page = audit_rows[AUDIT_START:AUDIT_START + 50]
if page:
    columns = list(page[0])
    header = "<tr><th>Row</th>" + "".join("<th>" + escape(k) + "</th>" for k in columns) + "</tr>"
    body = "".join("<tr><td>" + str(i) + "</td>" + "".join("<td>" + escape(row[k]) + "</td>" for k in columns) + "</tr>" for i, row in enumerate(page, AUDIT_START))
    display(HTML('<div style="overflow:auto;max-height:600px"><table>' + header + body + '</table></div>'))

def show_audit_images(index):
    row = audit_rows[index]
    print("Row:", index, "Edit:", row["modification"])
    for role in ("reference", "positive", "cf_a", "cf_b"):
        print(role, "|", row.get(role + "_caption", ""))
        with Image.open(image_root / row[role]) as im:
            preview = im.copy()
            preview.thumbnail((320, 320))
            display(preview)

show_audit_images(AUDIT_START)  # change index to inspect another quartet


### Xác nhận sau khi kiểm tra audit
Xem ít nhất 500 dòng và đối chiếu ảnh. Sau đó đổi giá trị bên dưới thành `True`, chạy lại ô xác nhận rồi tiếp tục huấn luyện. Không cần chạy lại mining.


In [ ]:
AUDIT_APPROVED = False  # set True only after reviewing the audit
if MODE == "screening" and not AUDIT_APPROVED:
    raise RuntimeError("Review mining_audit.csv, set AUDIT_APPROVED=True here, then rerun this cell.")


## 5. Tùy chọn chạy lại B0 theo 1K → 10K → 100K
B0 dùng Pic2Word gốc. Mỗi quy mô tạo run riêng, không trộn checkpoint.


In [ ]:
if MODE == "screening" and not AUDIT_APPROVED:
    raise RuntimeError("Complete the audit approval cell before training.")
import yaml
if RUN_B0_SCHEDULE:
    for sample_count in (1000, 10000, 100000):
        cfg = yaml.safe_load(Path("configs/train.yaml").read_text())
        cfg["training"].update(batch_size_per_device=4, max_steps=SETTINGS["steps"], seed=0, num_workers=0)
        cfg["data"].update(train_manifest=str(combined), image_root=str(image_root), max_samples=min(sample_count, len(rows)))
        root = runs_root / RUN_NAME / f"B0_{sample_count}_seed0"
        cfg["output"] = {"checkpoint_dir": str(root / "checkpoints"), "log_dir": str(root / "logs")}
        path = repo / f"configs/colab_b0_{sample_count}.yaml"; path.write_text(yaml.safe_dump(cfg))
        subprocess.run([sys.executable, "scripts/train.py", "--config", str(path), "--device", "cuda"], check=True)
else:
    print("Giữ checkpoint B0 hiện có. Bật RUN_B0_SCHEDULE nếu cần chạy lại.")


## 6. Thực nghiệm độc lập: B1, B4, B5
Mỗi biến thể dùng một cấu hình, checkpoint và thư mục riêng; dùng chung code train.
Mặc định `TRAIN_ENABLED=False`: đánh giá checkpoint đã có, không tự train lại.

**Nếu đã train trên Drive:** chạy ô cấu hình đầu notebook, ô nạp code, ô gắn Drive,
rồi chuyển thẳng tới đây. Không cần tải/mining dữ liệu lại. Chọn đúng `RUN_NAME` và `MODE`.
Nếu checkpoint ở vị trí khác, sửa `RUN_DIRS` bên dưới. Đường dẫn dữ liệu được đọc từ config
đã lưu cùng checkpoint; có thể ghi đè bằng `VAL_MANIFEST` và `EVAL_IMAGE_ROOT`.

**Nếu chưa train:** hoàn thành dữ liệu và audit trước, bật `TRAIN_ENABLED=True`,
rồi chạy riêng ô train B1, B4; đánh giá và kiểm tra GO trước khi train B5.
Kết quả chỉ có giá trị screening trên pseudo-label; không thay thế benchmark chính thức.

In [ ]:
from pathlib import Path
import csv, hashlib, json, subprocess, sys
import yaml
from IPython.display import display, HTML
from html import escape

TRAIN_ENABLED = False
EVAL_DEVICE = "cuda"
EVAL_BATCH_SIZE = 4
TOP_K = 100  # same shortlist size for all variants
RUN_DIRS = {v: runs_root / RUN_NAME / f"{v}_{MODE}_seed0" for v in ("B1", "B4", "B5")}
VAL_MANIFEST = None  # e.g. data_root / "pseudo_screening/val.csv"
EVAL_IMAGE_ROOT = None  # override if the saved image path has moved
COMPARISON_DIR = runs_root / RUN_NAME / "comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
CIRR_ROOT = data_root / "cirr"
FASHIONIQ_ROOT = data_root / "fashioniq"

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def saved_config(variant):
    return yaml.safe_load((RUN_DIRS[variant] / "config.yaml").read_text(encoding="utf-8"))

def eval_inputs(variant):
    cfg = saved_config(variant)
    manifest = Path(VAL_MANIFEST or Path(cfg["data"]["train_manifest"]).with_name("val.csv"))
    images = Path(EVAL_IMAGE_ROOT or cfg["data"]["image_root"])
    return manifest, images

def show_table(rows):
    if not rows:
        print("Chưa có kết quả thực nghiệm để so sánh.")
        return
    columns = list(dict.fromkeys(k for row in rows for k in row))
    head = "<tr>" + "".join(f"<th>{escape(k)}</th>" for k in columns) + "</tr>"
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(k, '—')))}</td>" for k in columns) + "</tr>" for row in rows)
    display(HTML('<div style="overflow:auto"><table>' + head + body + '</table></div>'))

readiness = []
for variant, directory in RUN_DIRS.items():
    missing = [name for name in ("best_checkpoint.pt", "config.yaml") if not (directory / name).is_file()]
    if not missing:
        manifest, images = eval_inputs(variant)
        if not manifest.is_file(): missing.append(str(manifest))
        if not images.is_dir(): missing.append(str(images))
    readiness.append({"variant": variant, "run_dir": str(directory),
                      "status": "Sẵn sàng đánh giá" if not missing else "Thiếu: " + ", ".join(missing)})
show_table(readiness)
(COMPARISON_DIR / "readiness.json").write_text(json.dumps(readiness, ensure_ascii=False, indent=2), encoding="utf-8")

def train_variant(variant):
    if not TRAIN_ENABLED:
        print(variant, "bỏ qua train; dùng checkpoint đã lưu nếu có.")
        return
    if MODE == "screening" and not AUDIT_APPROVED:
        raise RuntimeError("Hoàn thành kiểm tra audit trước khi train.")
    directory = RUN_DIRS[variant]
    if directory.exists() and any(directory.iterdir()):
        raise RuntimeError(f"Không ghi đè run cũ: {directory}. Chọn RUN_NAME mới.")
    cfg = yaml.safe_load(Path(f"configs/cfpe_{variant.lower()}.yaml").read_text())
    cfg["training"].update(batch_size_per_device=4, max_steps=SETTINGS["steps"], num_workers=0, seed=0)
    cfg["data"].update(train_manifest=str(pseudo_dir / "train.csv"), image_root=str(image_root))
    cfg["output"]["run_dir"] = str(directory)
    config_path = COMPARISON_DIR / f"train_{variant}.yaml"
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    subprocess.run([sys.executable, "-u", "scripts/train_cfpe.py", "--config", str(config_path),
                    "--device", EVAL_DEVICE], check=True)

### Train B1
Chạy riêng ô này; kết quả được lưu trong RUN_DIRS["B1"].

In [ ]:
train_variant("B1")

### Train B4
Chạy riêng ô này; kết quả được lưu trong RUN_DIRS["B4"].

In [ ]:
train_variant("B4")

## 7. Đánh giá counterfactual B1 và B4
Chạy cùng tập validation; lưu ở `evaluation_cf/`, tách khỏi log train.
Thiếu checkpoint hoặc dữ liệu sẽ báo rõ, không tạo tỷ lệ giả.

In [ ]:
def evaluate_cf(variant):
    directory = RUN_DIRS[variant]
    checkpoint = directory / "best_checkpoint.pt"
    if not checkpoint.is_file() or not (directory / "config.yaml").is_file():
        print(variant, "chưa có checkpoint/config:", directory)
        return
    manifest, images = eval_inputs(variant)
    if not manifest.is_file() or not images.is_dir():
        print(variant, "thiếu dữ liệu:", manifest, images)
        return
    destination = directory / "evaluation_cf"
    subprocess.run([sys.executable, "scripts/evaluate_counterfactual.py",
        "--manifest", str(manifest), "--image-root", str(images), "--checkpoint", str(checkpoint),
        "--run-dir", str(destination), "--batch-size", str(EVAL_BATCH_SIZE), "--device", EVAL_DEVICE], check=True)
    cfg = saved_config(variant)
    protocol = {"manifest_sha256": file_hash(manifest), "checkpoint_sha256": file_hash(checkpoint),
                "image_root": str(images.resolve()), "training": cfg["training"], "model": cfg["model"]}
    (destination / "protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")

evaluate_cf("B1")
evaluate_cf("B4")

## 8. GO gate, train và đánh giá B5
GO khi B4 tăng both-win và giảm hard-negative error so với B1 trên cùng validation.
Hai chỉ số này là phần bù của nhau trong evaluator hiện tại, không phải hai bằng chứng độc lập.
Checkpoint B5 đã có vẫn có thể được đánh giá; gate chỉ điều khiển việc train B5 mới và benchmark.

In [ ]:
def read_cf(variant):
    root = RUN_DIRS[variant] / "evaluation_cf"
    if not (root / "metrics.json").is_file() or not (root / "protocol.json").is_file():
        return None
    protocol = json.loads((root / "protocol.json").read_text())
    checkpoint = RUN_DIRS[variant] / "best_checkpoint.pt"
    manifest, images = eval_inputs(variant)
    if (not checkpoint.is_file() or not manifest.is_file()
        or protocol["checkpoint_sha256"] != file_hash(checkpoint)
        or protocol["manifest_sha256"] != file_hash(manifest)
        or protocol["image_root"] != str(images.resolve())):
        raise RuntimeError(f"{variant}: kết quả cũ không khớp dữ liệu/checkpoint; chạy lại đánh giá.")
    return json.loads((root / "metrics.json").read_text())["counterfactual_evaluation"], protocol

def comparable_protocol(left, right):
    if (left["manifest_sha256"], left["image_root"]) != (right["manifest_sha256"], right["image_root"]):
        return False
    for key in ("seed", "batch_size_per_device", "max_steps", "epochs", "learning_rate", "weight_decay", "precision"):
        if left["training"].get(key) != right["training"].get(key): return False
    return all(left["model"].get(k) == right["model"].get(k)
               for k in ("backbone", "pretrained", "slots", "hidden_dim"))

def go_gate():
    b1, b4 = read_cf("B1"), read_cf("B4")
    if b1 is None or b4 is None:
        print("Chưa đủ kết quả B1/B4 để xác định GO.")
        return False
    if not comparable_protocol(b1[1], b4[1]):
        raise RuntimeError("B1/B4 khác dữ liệu hoặc điều kiện train; chưa thể kết luận GO.")
    m1, m4 = b1[0], b4[0]
    go = (m4["both_win_percent"] > m1["both_win_percent"] and
          m4["hard_negative_error_percent"] < m1["hard_negative_error_percent"])
    print("GO:", go, "| B1 both-win:", m1["both_win_percent"], "| B4:", m4["both_win_percent"])
    return go

go = go_gate()

In [ ]:
if TRAIN_ENABLED and RUN_B5_AFTER_GO and go_gate():
    train_variant("B5")
else:
    print("Không train B5 mới: train chưa bật hoặc chưa đạt GO.")

In [ ]:
evaluate_cf("B5")

## 9. Benchmark cùng điều kiện cho cả B1, B4, B5
Chạy sau khi đạt GO và có đủ dữ liệu chính thức. Cùng tập validation, kho ảnh và Top-K=100.
Chỉ tổng hợp global recall CIRR; group recall từ script hiện tại lọc từ global Top-K,
không phải đánh giá đầy đủ nhóm ứng viên, nên không dùng để kết luận chính thức.
Kết quả trước/sau rerank B5 được khôi phục từ điểm đã lưu trên cùng shortlist.

In [ ]:
if go_gate():
    for variant, directory in RUN_DIRS.items():
        checkpoint = directory / "best_checkpoint.pt"
        if not checkpoint.is_file():
            print(variant, "thiếu checkpoint; bỏ qua.")
            continue
        if CIRR_ROOT.is_dir():
            subprocess.run([sys.executable, "scripts/evaluate_cfpe_cirr.py", "--dataset-root", str(CIRR_ROOT),
                "--checkpoint", str(checkpoint), "--index", str(data_root / "indexes/cirr_val.pt"),
                "--run-dir", str(directory / "cirr"), "--top-k", str(TOP_K), "--device", EVAL_DEVICE], check=True)
            (directory / "cirr/protocol.json").write_text(json.dumps({"checkpoint_sha256": file_hash(checkpoint), "dataset_root": str(CIRR_ROOT.resolve()), "top_k": TOP_K}), encoding="utf-8")
        else: print("Chưa có CIRR:", CIRR_ROOT)
        if FASHIONIQ_ROOT.is_dir():
            subprocess.run([sys.executable, "scripts/evaluate_cfpe_fashioniq.py", "--dataset-root", str(FASHIONIQ_ROOT),
                "--checkpoint", str(checkpoint), "--index-dir", str(data_root / "indexes"),
                "--run-dir", str(directory / "fashioniq"), "--top-k", str(TOP_K), "--device", EVAL_DEVICE], check=True)
            (directory / "fashioniq/protocol.json").write_text(json.dumps({"checkpoint_sha256": file_hash(checkpoint), "dataset_root": str(FASHIONIQ_ROOT.resolve()), "top_k": TOP_K}), encoding="utf-8")
        else: print("Chưa có Fashion-IQ:", FASHIONIQ_ROOT)
else:
    print("Chưa chạy benchmark: chưa đạt GO hoặc thiếu đánh giá B1/B4.")

## 10. So sánh từ kết quả đã lưu — không train hoặc đánh giá lại
CF win và recall tính bằng %. Chênh lệch là **điểm phần trăm**.
CF sau verifier chỉ đo trên bộ ba positive/CF-A/CF-B, không phải recall toàn kho ảnh.
Không so sánh tổng training loss vì các biến thể có mục tiêu loss khác nhau.
Một seed chỉ cho kết quả ban đầu; chưa chứng minh cải thiện ổn định.

In [ ]:
import math
comparison = []
available = {}
for variant in RUN_DIRS:
    result = read_cf(variant)
    if result is None:
        print(variant, "chưa có kết quả đánh giá.")
        continue
    metrics, protocol = result
    available[variant] = (metrics, protocol)
    comparison.append({"variant": variant, "evaluation": "CF stage1",
        "queries": metrics["counterfactual_query_count"],
        "CF-A win %": metrics["cf_a_win_percent"], "CF-B win %": metrics["cf_b_win_percent"],
        "both win %": metrics["both_win_percent"], "hard-negative error %": metrics["hard_negative_error_percent"]})
    if variant == "B5":
        records = json.loads((RUN_DIRS[variant] / "evaluation_cf/constraint_scores.json").read_text())
        wins_a, wins_b = [], []
        for record in records:
            scores = {}
            for role, score_key in (("positive", "positive_score"), ("cf_a", "cf_a_score"), ("cf_b", "cf_b_score")):
                p, e, v = record["verifier"][role]
                scores[role] = record[score_key] + .5 * math.log(max(p, 1.1920929e-7)) + .5 * math.log(max(e, 1.1920929e-7)) - .5 * math.log(max(v, 1.1920929e-7))
            wins_a.append(scores["positive"] > scores["cf_a"])
            wins_b.append(scores["positive"] > scores["cf_b"])
        n = len(records)
        if n:
            both = 100 * sum(a and b for a, b in zip(wins_a, wins_b)) / n
            comparison.append({"variant": variant, "evaluation": "CF verifier (3 candidates)", "queries": n,
                "CF-A win %": 100 * sum(wins_a) / n, "CF-B win %": 100 * sum(wins_b) / n,
                "both win %": both, "hard-negative error %": 100 - both})

for left, right in (("B1", "B4"), ("B4", "B5")):
    if left in available and right in available:
        a, pa = available[left]; b, pb = available[right]
        if comparable_protocol(pa, pb):
            print(f"{right} - {left}, stage1 both-win: {b['both_win_percent'] - a['both_win_percent']:+.2f} điểm phần trăm")
        else:
            print(left, right, "khác điều kiện; không kết luận cải thiện từ hai run này.")

show_table(comparison)
(COMPARISON_DIR / "counterfactual_comparison.json").write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
if comparison:
    columns = list(dict.fromkeys(k for row in comparison for k in row))
    with (COMPARISON_DIR / "counterfactual_comparison.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader(); writer.writerows(comparison)
print("Đã lưu bảng tại:", COMPARISON_DIR)

In [ ]:
from collections import defaultdict
benchmark_rows = []
benchmark_protocols = {}
for variant, directory in RUN_DIRS.items():
    for dataset, folder, prefix, ks in (("CIRR", "cirr", "", (1, 5, 10, 50)),
                                       ("Fashion-IQ", "fashioniq", "fashioniq_", (10, 50))):
        pred_path = directory / folder / f"{prefix}per_query_predictions.json"
        score_path = directory / folder / f"{prefix}constraint_scores.json"
        if not pred_path.is_file() or not score_path.is_file():
            continue
        protocol_path = directory / folder / "protocol.json"
        if not protocol_path.is_file():
            print(variant, dataset, "thiếu metadata kiểm chứng; chạy lại ô benchmark.")
            continue
        protocol = json.loads(protocol_path.read_text())
        if protocol["checkpoint_sha256"] != file_hash(directory / "best_checkpoint.pt"):
            raise ValueError(f"{variant}/{dataset}: checkpoint đã đổi; cần đánh giá lại.")
        predictions = json.loads(pred_path.read_text())
        constraints = json.loads(score_path.read_text())
        if not predictions or len(predictions) != len(constraints):
            raise ValueError(f"{variant}/{dataset}: dự đoán và điểm không khớp.")
        signature = [(p.get("pair_id"), p.get("category"), p.get("reference_id"), p["target_id"], p.get("caption"), p.get("captions")) for p in predictions]
        identity = (protocol["dataset_root"], protocol["top_k"], signature)
        if dataset in benchmark_protocols and benchmark_protocols[dataset] != identity:
            raise ValueError(f"{dataset}: khác query, dữ liệu hoặc Top-K giữa các biến thể.")
        benchmark_protocols[dataset] = identity
        for stage in (("stage1", "rerank") if variant == "B5" else ("stage1",)):
            groups = defaultdict(list)
            for pred, score in zip(predictions, constraints):
                key = "pair_id" if dataset == "CIRR" else "reference_id"
                if pred[key] != score[key] or pred.get("category") != score.get("category"):
                    raise ValueError("Sai thứ tự query trong artifact.")
                field = "stage1_score" if stage == "stage1" else "final_score"
                ranking = [r["image_id"] for r in sorted(score["top_k"], key=lambda r: r[field], reverse=True)]
                groups[pred.get("category", "all")].append((pred["target_id"], ranking))
            row = {"variant": variant, "dataset": dataset, "stage": stage, "queries": len(predictions)}
            for k in ks:
                values = [100 * sum(target in ranking[:k] for target, ranking in items) / len(items) for items in groups.values()]
                row[f"Recall@{k} %"] = sum(values) / len(values)
            benchmark_rows.append(row)
show_table(benchmark_rows)
(COMPARISON_DIR / "benchmark_comparison.json").write_text(json.dumps(benchmark_rows, ensure_ascii=False, indent=2), encoding="utf-8")
if benchmark_rows:
    columns = list(dict.fromkeys(k for row in benchmark_rows for k in row))
    with (COMPARISON_DIR / "benchmark_comparison.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader(); writer.writerows(benchmark_rows)